In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser , PydanticOutputParser

In [2]:
from pydantic import BaseModel 
from typing import Literal
class llm_schema(BaseModel):
    movie_summary_flag : Literal["positive", "negitive"]  
    

### Chain with Conitional chains

In [3]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You Are A movie review evaluater"), 
        ("human", "please categorize the movie review as positive or negitive: {input}")       
    ]
)

In [4]:
llm = ChatGroq(model = "qwen/qwen3-32b")

In [5]:
llm_structured_op = llm.with_structured_output(llm_schema)


In [6]:
from langchain_core.runnables import RunnableLambda

def pydantic_json(input : llm_schema) ->str:
    
    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

### Conditional Chain 1

In [7]:
linkedin_post = ChatPromptTemplate.from_messages(
    [
        ("system" , "You are a smart linked in post generator"),
        ("human" , "geneate a post for the linkedin for given text : {text}")
    ]
)

llm = ChatGroq(model = "qwen/qwen3-32b")

str_parser = StrOutputParser()

linkedin_chain = linkedin_post | llm | str_parser


### Conditional chain 2

In [8]:
from langchain_core.runnables import RunnableLambda , RunnableBranch

In [13]:
def insta_chain(data: dict):
    
    insta_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a smart Instagram post generator"),
            ("human", "Generate a post for Instagram for the given text:\n{text}")
        ]
    )

    llm = ChatGroq(model="qwen/qwen3-32b")

    parser = StrOutputParser()

    chain = insta_prompt | llm | parser

    return chain.invoke(data)


insta_chain_runnable = RunnableLambda(insta_chain)

### Final Orchestration

In [14]:
conditional_chain = RunnableBranch(
    (lambda x : "positive" in x , linkedin_chain),
    insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_op | pydantic_json_lambda | conditional_chain


In [16]:
final_orchestrator.invoke({"input":"i hate this kgf movie"})

'<think>\nOkay, the user wants me to generate an Instagram post based on the given text "negitive". First, I need to check if there\'s a typo. "Negitive" is probably a misspelling of "negative". So I should consider that.\n\nNow, the user didn\'t provide much context. They just said "negitive". Maybe they want a post about dealing with negativity, or perhaps they want to highlight the positive. Since Instagram is a visual platform, the post should be engaging and relatable.\n\nI should start by addressing the negative theme but flip it into something positive. Maybe use phrases like "turn negatives into positives" to encourage a positive mindset. Including emojis like 🌧️ for the negative cloud and ☀️ for the positive sun makes it visually appealing.\n\nAlso, a call to action is important. Adding a question like "What\'s one thing you\'re grateful for today?" can engage the audience. The hashtag #NegativeToPositive adds a catchy touch. I need to make sure the message is uplifting and mo